# Training YOLOv8 Smart Gate

Notebook ini memakai `dataset_merged/` sebagai dataset utama. `plat-nomor/` digunakan sebagai fallback jika dataset gabungan tidak tersedia.

In [7]:
!pip install -q ultralytics pyyaml gdown

from pathlib import Path
import zipfile
import gdown
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU CUDA tidak aktif. Di Colab pilih Runtime > Change runtime type > T4 GPU lalu jalankan ulang.')
print('GPU:', torch.cuda.get_device_name(0))

# Dataset gabungan diunduh langsung dari Google Drive.
FILE_ID = '138RROLwz9wkpmgJiwwYOl3Tf1UahDKZz'
ZIP_PATH = Path('/content/smartgate-all-datasets.zip')
if not ZIP_PATH.is_file():
    gdown.download(id=FILE_ID, output=str(ZIP_PATH), quiet=False)
if not ZIP_PATH.is_file() or ZIP_PATH.stat().st_size < 1000000:
    raise RuntimeError(f'Download ZIP gagal atau file terlalu kecil: {ZIP_PATH}')
with zipfile.ZipFile(ZIP_PATH) as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f'ZIP rusak pada entry: {bad_member}')
    archive.extractall('/content')

candidates = [path.parent for path in Path('/content').rglob('data.yaml') if path.parent.name in ('dataset_merged', 'plat-nomor')]
preferred = [path for path in candidates if path.name == 'dataset_merged']
DATA_ROOT = next(iter(preferred or candidates), None)
if DATA_ROOT is None:
    raise FileNotFoundError('data.yaml tidak ditemukan setelah download dan ekstraksi ZIP.')
DATA_YAML = DATA_ROOT / 'data.yaml'
print('ZIP:', ZIP_PATH, f'({ZIP_PATH.stat().st_size / 1024**3:.2f} GB)')
print('Dataset:', DATA_ROOT)
print('Config sumber:', DATA_YAML)

GPU: Tesla T4
ZIP: /content/smartgate-all-datasets.zip (2.22 GB)
Dataset: /content/dataset_merged
Config sumber: /content/dataset_merged/data.yaml


In [8]:
import yaml

config = yaml.safe_load(DATA_YAML.read_text())
config['path'] = str(DATA_ROOT)
config['train'] = 'train/images'
config['val'] = 'valid/images'
config['test'] = 'test/images'
config['nc'] = 1
config['names'] = ['number-plate']
RUNTIME_YAML = Path('/content/dataset_runtime.yaml')
RUNTIME_YAML.write_text(yaml.safe_dump(config, sort_keys=False))
print(RUNTIME_YAML.read_text())

for split in ('train', 'val', 'test'):
    image_dir = DATA_ROOT / config[split]
    label_dir = DATA_ROOT / config[split].replace('/images', '/labels')
    if not image_dir.is_dir():
        raise FileNotFoundError(f'{split} images tidak ditemukan: {image_dir}')
    if not label_dir.is_dir():
        raise FileNotFoundError(f'{split} labels tidak ditemukan: {label_dir}')
    images = {p.stem for p in image_dir.iterdir() if p.is_file()}
    labels = {p.stem for p in label_dir.glob('*.txt')}
    missing_labels = sorted(images - labels)
    orphan_labels = sorted(labels - images)
    if missing_labels or orphan_labels:
        raise ValueError(f'{split}: missing labels={len(missing_labels)}, orphan labels={len(orphan_labels)}')
    for label_path in label_dir.glob('*.txt'):
        for line_no, line in enumerate(label_path.read_text().splitlines(), 1):
            fields = line.split()
            if len(fields) != 5 or int(fields[0]) != 0 or not all(0.0 <= float(v) <= 1.0 for v in fields[1:]):
                raise ValueError(f'Anotasi invalid: {label_path}:{line_no} -> {line}')
    print(f'{split}: {len(images)} images, {len(labels)} labels, valid')
print('Struktur dan anotasi dataset valid.')

path: /content/dataset_merged
train: train/images
val: valid/images
test: test/images
nc: 1
names:
- number-plate

train: 1572 images, 1572 labels, valid
val: 290 images, 290 labels, valid
test: 84 images, 84 labels, valid
Struktur dan anotasi dataset valid.


In [9]:
def count_labels(folder):
    files = list(Path(folder).glob('*.txt'))
    objects = sum(len(path.read_text().splitlines()) for path in files)
    return len(files), objects

for split in ('train', 'valid', 'test'):
    label_dir = DATA_ROOT / split / 'labels'
    files, objects = count_labels(label_dir)
    print(f'{split}: {files} file label, {objects} anotasi')

train: 1572 file label, 2284 anotasi
valid: 290 file label, 446 anotasi
test: 84 file label, 102 anotasi


## Training

Gunakan GPU Colab: **Runtime > Change runtime type > T4 GPU**. Kotak anotasi harus tepat mengelilingi papan plat, bukan bumper, kamera, gril, atau seluruh kendaraan.

In [10]:
from ultralytics import YOLO
import torch

device = 0
batch = 16
print('Device:', device)

RUN_DIR = DATA_ROOT.parent / 'runs' / 'detect'
model = YOLO('yolov8n.pt')
model.train(
    data=str(RUNTIME_YAML),
    epochs=50,
    imgsz=640,
    batch=batch,
    device=device,
    project=str(RUN_DIR),
    name='smartgate-colab',
    patience=12,
    hsv_h=0.015,
    hsv_s=0.55,
    hsv_v=0.35,
    degrees=5.0,
    translate=0.08,
    scale=0.35,
    fliplr=0.0,
    mosaic=0.7,
    mixup=0.05,
    close_mosaic=10,
    pretrained=True,
    plots=True,
    val=True,
    seed=42,
    workers=2,
    cache=False,
    exist_ok=True,
)
print('Training selesai.')

Device: 0
Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_runtime.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.55, hsv_v=0.35, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.7, multi_scale=0.0, name=smartgate-colab, nbs=64, nms=None, opset=None, 

In [11]:
best_pt = RUN_DIR / 'smartgate-colab' / 'weights' / 'best.pt'
if not best_pt.is_file():
    raise FileNotFoundError(best_pt)
trained = YOLO(str(best_pt))
val_metrics = trained.val(data=str(RUNTIME_YAML), split='val', imgsz=640, device=device)
test_metrics = trained.val(data=str(RUNTIME_YAML), split='test', imgsz=640, device=device)
print('best.pt:', best_pt)
print(f'Validation mAP50: {val_metrics.box.map50:.4f}')
print(f'Validation mAP50-95: {val_metrics.box.map:.4f}')
print(f'Test mAP50: {test_metrics.box.map50:.4f}')
print(f'Test mAP50-95: {test_metrics.box.map:.4f}')

Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2115.6±1426.6 MB/s, size: 474.3 KB)
val: Scanning /content/dataset_merged/valid/labels.cache... 290 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 290/290 110.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 3.6it/s 5.3s0.2s
                   all        290        446      0.822       0.85      0.872      0.508
Speed: 2.0ms preprocess, 4.6ms inference, 0.0ms loss, 2.5ms postprocess per image
Results saved to /content/runs/detect/val-3
Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 85.2±60.7 MB/s, size: 231.8 KB)
val: Scanning /con

In [12]:
print('Bobot selesai:', best_pt)
print('File model tersedia:', best_pt.is_file())

Bobot selesai: /content/runs/detect/smartgate-colab/weights/best.pt
Download best.pt untuk deployment lokal:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
import json

test_map50 = float(test_metrics.box.map50)
test_map5095 = float(test_metrics.box.map)
val_map50 = float(val_metrics.box.map50)
report = {
    'model': str(best_pt),
    'validation': {
        'mAP50': val_map50,
        'mAP50_95': float(val_metrics.box.map),
    },
    'test': {
        'images': 84,
        'instances': 102,
        'mAP50': test_map50,
        'mAP50_95': test_map5095,
    },
    'detector_target_mAP50': 0.98,
    'detector_target_met': test_map50 >= 0.98,
    'production_kpi_status': 'NOT_CLAIMED',
}
REPORT_PATH = RUN_DIR / 'smartgate-colab' / 'evaluation_summary.json'
REPORT_PATH.write_text(json.dumps(report, indent=2))
print('=== LAPORAN EVALUASI DETECTOR ===')
print(f'Validation mAP50    : {val_map50:.4f} ({val_map50:.2%})')
print(f'Validation mAP50-95 : {float(val_metrics.box.map):.4f}')
print(f'Test mAP50          : {test_map50:.4f} ({test_map50:.2%})')
print(f'Test mAP50-95       : {test_map5095:.4f}')
print(f'Target test mAP50   : 0.9800 (98.00%)')
print('Target detector     : ' + ('TERCAPAI' if report['detector_target_met'] else 'BELUM TERCAPAI'))
print('KPI produksi        : BELUM DIKLAIM (perlu evaluasi OCR, fusion, false accept, dan latency)')
print('Ringkasan tersimpan :', REPORT_PATH)
print('Laporan tersedia:', REPORT_PATH.is_file())

=== LAPORAN EVALUASI DETECTOR ===
Validation mAP50    : 0.8723 (87.23%)
Validation mAP50-95 : 0.5078
Test mAP50          : 0.9745 (97.45%)
Test mAP50-95       : 0.5870
Target test mAP50   : 0.9800 (98.00%)
Target detector     : BELUM TERCAPAI
KPI produksi        : BELUM DIKLAIM (perlu evaluasi OCR, fusion, false accept, dan latency)
Ringkasan tersimpan : /content/runs/detect/smartgate-colab/evaluation_summary.json
Download evaluation_summary.json untuk dokumentasi hasil:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
from google.colab import files

ARTIFACT_DIR = RUN_DIR / 'smartgate-colab'
required_files = [best_pt, REPORT_PATH]
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError('Artefak belum tersedia: ' + ', '.join(missing))

DOWNLOAD_DIR = Path('/content/smartgate-artifacts')
DOWNLOAD_DIR.mkdir(exist_ok=True)
shutil.copy2(best_pt, DOWNLOAD_DIR / 'best.pt')
shutil.copy2(REPORT_PATH, DOWNLOAD_DIR / 'evaluation_summary.json')
for result_name in ('results.csv', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png'):
    result_path = ARTIFACT_DIR / result_name
    if result_path.is_file():
        shutil.copy2(result_path, DOWNLOAD_DIR / result_name)

ARCHIVE_PATH = Path('/content/smartgate-artifacts.zip')
if ARCHIVE_PATH.exists():
    ARCHIVE_PATH.unlink()
shutil.make_archive(str(ARCHIVE_PATH.with_suffix('')), 'zip', DOWNLOAD_DIR)
if not ARCHIVE_PATH.is_file() or ARCHIVE_PATH.stat().st_size == 0:
    raise RuntimeError(f'ZIP artefak gagal dibuat: {ARCHIVE_PATH}')
print('Artefak siap diunduh:', ARCHIVE_PATH)
print(f'Ukuran ZIP: {ARCHIVE_PATH.stat().st_size / 1024**2:.2f} MB')
print('Jika download tidak muncul, buka panel Files Colab dan klik kanan smartgate-artifacts.zip > Download.')
files.download(str(ARCHIVE_PATH))

Setelah training selesai, salin `best.pt` ke:

`Tugas Citra Digital/prototype/runs/detect/smartgate-colab/weights/best.pt`

Lalu jalankan aplikasi dengan `SGS_YOLO_WEIGHTS` menunjuk ke file tersebut dan restart Flask.